In [38]:
import pandas as pd

In [39]:
df = pd.read_csv("../data/05_model_input/downloaded_scenarios.csv")
df = df.loc[df["sector"] == "Power",:]

/var/folders/df/zghzv05d7xb8t9xy7y5ld_h40000gn/T/ipykernel_23570/3532604135.py:1: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/05_model_input/downloaded_scenarios.csv")


# Compatible scenarios

In [40]:
# the EBITDA is likely to be negative when the values in fom_usd_per_mw_yr are higher
#  than capacity_factor * hours_per_year * scenario_price
incompatible_fixed_cost = df["om_cost_usd_per_mw_per_yr"] > df["scenario_capacity_factor"] * (24*365) * df["scenario_price"]

# the EBITDA is likely to be negative when fuel_price/efficiency > scenario_price .
incompatible_var_cost = df["fuel_price"] / df["efficiency_decimal"] > df["scenario_price"]

In [41]:
df = df.assign(
    incompatible_var_cost = incompatible_var_cost.astype(bool),
    incompatible_fixed_cost = incompatible_fixed_cost.astype(bool),
)



In [42]:
df.loc[df["scenario_type"] == "baseline", ["scenario_provider", "scenario"]].drop_duplicates()

,scenario_provider,scenario
368114,IMAGE 3.2,SSP1-baseline
399793,IMAGE 3.2,SSP2-baseline
1208657,WITCH 5.0,CO_CurPol
1349111,WITCH 5.0,EN_NoPolicy


In [50]:
pd.options.display.max_rows = 1000

incompatibility_flagged = df[[
    "scenario_provider", "scenario", "scenario_type","sector", "technology",  "scenario_geography",
    "incompatible_var_cost","incompatible_fixed_cost"]].drop_duplicates()


incompatibility_flagged = incompatibility_flagged.groupby(["scenario_provider", "scenario"]).agg(
    n_techs = ("technology", "nunique"),
    n_regions = ("scenario_geography", "nunique"),
    incompatible_var_cost = ("incompatible_var_cost", "any"),
    incompatible_fixed_cost = ("incompatible_fixed_cost", "any"),
)

incompatibility_flagged = incompatibility_flagged.assign(
    likely_incompatible_scenario = (incompatibility_flagged["incompatible_var_cost"] | incompatibility_flagged["incompatible_fixed_cost"]),
    incompatible_scenario = (incompatibility_flagged["incompatible_var_cost"] & incompatibility_flagged["incompatible_fixed_cost"]),
)


incompatibility_flagged.query("incompatible_var_cost == False")

n_techs  \
scenario_provider     scenario                                                 
AIM/CGE-Korea 2.1     CO_2Deg2020                                         11   
                      CO_2Deg2030                                         11   
                      CO_BAU                                               8   
                      CO_Bridge                                           11   
                      CO_CurPol                                            8   
                      CO_GPP                                              11   
                      CO_NDCMCS                                           11   
AIM/Enduse-Japan 2.1  EN_NP_2025_-1002050                                  9   
                      EN_NP_2025_-502050                                   8   
                      EN_NP_2025_-602050                                   9   
                      EN_NP_2025_-702050                                   9   
                      EN_NP_2025_-802050                                   9   
                      EN_NP_2025_-902050                                   9   
                      EN_NP_BL                                             8   
                      EN_NP_CurPol                                         9   
                      EN_NP_UNDC                                           8   
AIM/Hub-Japan 2.1     EN_NP_2025_-1002050                                 11   
                      EN_NP_2025_-302050                                  10   
                      EN_NP_2025_-402050                                  10   
                      EN_NP_2025_-502050                                  11   
                      EN_NP_2025_-602050                                  11   
                      EN_NP_2025_-702050                                  11   
                      EN_NP_2025_-802050                                  11   
                      EN_NP_2025_-902050                                  11   
                      EN_NP_CurPol                                         8   
                      EN_NP_UNDC                                          10   
AIM/Hub-Korea 2.0     EN_NP_2025_-1002050                                 11   
                      EN_NP_2025_-302050                                   8   
                      EN_NP_2025_-402050                                   9   
                      EN_NP_2025_-502050                                  11   
                      EN_NP_2025_-602050                                  11   
                      EN_NP_2025_-702050                                  10   
                      EN_NP_2025_-802050                                  11   
                      EN_NP_2025_-902050                                  10   
                      EN_NP_CurPol                                         8   
                      EN_NP_UNDC                                          10   
COFFEE 1.1            CO_2Deg2020                                          9   
                      CO_2Deg2030                                          9   
                      CO_BAU                                               9   
                      CO_Bridge                                            9   
                      CO_CurPol                                            9   
                      CO_GPP                                               9   
                      CO_NDCplus                                           9   
                      EN_INDCi2030_1000                                    9   
                      EN_INDCi2030_1000_NDCp                               9   
                      EN_INDCi2030_1000f                                   9   
                      EN_INDCi2030_1200                                    9   
                      EN_INDCi2030_1200f                                   9   
                      EN_INDCi2030_1400                              

In [49]:
pd.options.display.max_rows = 1000

incompatibility_flagged = df[[
    "scenario_provider", "scenario", "scenario_type","sector", "technology",   "scenario_geography",
    "incompatible_var_cost","incompatible_fixed_cost"]].drop_duplicates()


incompatibility_flagged = incompatibility_flagged.groupby(["scenario_provider", "scenario"]).agg(
    n_techs = ("technology", "nunique"),
    n_regions = ("scenario_geography", "nunique"),
    incompatible_var_cost = ("incompatible_var_cost", "sum"),
    incompatible_fixed_cost = ("incompatible_fixed_cost", "sum"),
).assign(
    n_tech_regions = lambda x: x["n_techs"] * x["n_regions"]
)
print(incompatibility_flagged.shape)
incompatibility_flagged.query("incompatible_fixed_cost < n_tech_regions/2")

(1030, 5)


n_techs  n_regions  \
scenario_provider scenario                                 
AIM/CGE 2.2       EN_INDCi2030_1000f       14         18   
                  EN_INDCi2030_1200        14         18   
                  EN_INDCi2030_1200f       14         18   
                  EN_INDCi2030_1400        14         18   
                  EN_INDCi2030_1400f       14         18   
...                                       ...        ...   
WITCH 5.0         EN_NPi2020_800f          15         20   
                  EN_NPi2020_900           15         20   
                  EN_NPi2020_900f          15         20   
                  EN_NPi2100               15         20   
                  EN_NoPolicy              15         20   

                                      incompatible_var_cost  \
scenario_provider scenario                                    
AIM/CGE 2.2       EN_INDCi2030_1000f                      3   
                  EN_INDCi2030_1200                       3   
                  EN_INDCi2030_1200f                      3   
                  EN_INDCi2030_1400                       3   
                  EN_INDCi2030_1400f                      3   
...                                                     ...   
WITCH 5.0         EN_NPi2020_800f                       227   
                  EN_NPi2020_900                        229   
                  EN_NPi2020_900f                       226   
                  EN_NPi2100                            137   
                  EN_NoPolicy                           132   

                                      incompatible_fixed_cost  n_tech_regions  
scenario_provider scenario                                                     
AIM/CGE 2.2       EN_INDCi2030_1000f                       10             252  
                  EN_INDCi2030_1200                         5             252  
                  EN_INDCi2030_1200f                        4             252  
                  EN_INDCi2030_1400                         4             252  
                  EN_INDCi2030_1400f                        4             252  
...                                                       ...             ...  
WITCH 5.0         EN_NPi2020_800f                         121             300  
                  EN_NPi2020_900                          124             300  
                  EN_NPi2020_900f                         116             300  
                  EN_NPi2100                               80             300  
                  EN_NoPolicy                              66             300  

[1027 rows x 5 columns]